In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sys
sys.path.append('..')  # Go up one level
from amplifier import Amplifier

# Example: An amplifier

The data is taken from the amplifier of Figure 4 in reference \[1\], but the impedance matrix $\mathbf{Z}$ elements are a little different from Table 1 because of minor design differences.

References: 
1. W. Wang, V. Zhurbenko, J. D. Sánchez-Heredia, and J. H. Ardenkjær-Larsen. “Trade-off between preamplifier noise figure and decoupling in MRI detectors”. In: Magnetic Resonance in Medicine 89 (2 Feb. 2023), pp. 859–871. ISSN: 15222594. DOI: 10.1002/mrm.29489. 

↓----- The parameters below can be changed -----↓

⚠️**Note**: if you open an S parameter file in a text editor and read the noise parameters, the $R_n$ readings have already been divided by 50. Do not  copy a number of $R_n$ directly from the text editor; multiply it by 50 Ω and put the product here!

In [ ]:
ar_s = np.array([
    [0.8526421-0.07518526j, 0.001411245+0.00340048j], 
    [-13.79295+6.784279j,   0.1176207-0.5604914j]
]) # S parameters, reference 50 Ω
s_opt = 0.2564734+0.03810154j  # s_opt, reference 50 Ω
nf_min_dB = 0.5832359  # dB
rn = 5.616032  # Rn, Ω. Do not forget to multiple by 50 Ω when taken fresh from an S2P file!
amp = Amplifier(ar_s, nf_min_dB, s_opt, rn)

# --- dNF vector, i.e. excess noise figure above the minimum in dB ----------------
dNF = np.concatenate([
    np.arange(0,    0.030 + 0.001, 0.001),
    [0.035, 0.045],
    np.arange(0.05, 0.1   + 0.01,  0.01),
    np.arange(0.12, 0.5   + 0.02,  0.02),
    np.arange(0.55, 1.3   + 0.05,  0.05),
])

↑----- The parameters above can be changed -----↑

## Results
Two plots: Maximum decoupling vs NF, maximum gain vs NF

Below we set the amplifier input impedance **varying**; that means we present Z_{out} to the amplifier, then put a load at the amplifier output to get the maximum small-signal power. This will change the amplifier input impedance slightly, and we use this amplifier impedance to calculate $\Gamma_\mathrm{out}$, pramplifier decoupling, and maximum gain. This is the original version in §2.2 of reference \[1\].

In [ ]:
# --- compute ---------------------------------------------------------------
min_I, _, _, max_G, _, _, _, _ = amp.min_and_max_I_and_G_at_given_NF(dNF)  # min_I is maximum preamplifier decoupling; see eq. (16) in reference [1].

# --- plot ------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, num=1, clear=True)

(line1,) = ax1.plot(dNF + nf_min_dB, -20 * np.log10(min_I), linewidth=1, label="pramp decoupling")

ax1.plot(nf_min_dB, -20 * np.log10(min_I[0]),
        marker=".", markersize=15, color=line1.get_color(), label="_nolegend_")

ax1.annotate(
    f" {nf_min_dB:.2f} dB NF\n {-20 * np.log10(min_I[0]):.2f} dB Decoupling",
    xy=(nf_min_dB, -20 * np.log10(min_I[0])),
    fontfamily="monospace", fontsize=12, va="center", ha="left",
)  # Put a marker on the head

(line2,) = ax2.plot(dNF + nf_min_dB, 10 * np.log10(max_G), linewidth=1, label="max relative gain")

ax2.plot(nf_min_dB, 10 * np.log10(max_G[0]),
        marker=".", markersize=15, color=line1.get_color(), label="_nolegend_")

ax2.annotate(
    f" {nf_min_dB:.2f} dB NF\n {10 * np.log10(max_G[0]):.2f} dB Gain",
    xy=(nf_min_dB, 10 * np.log10(max_G[0])),
    fontfamily="monospace", fontsize=12, va="center", ha="left",
)  # Put a marker on the head

# --- figure formatting -----------------------------------------------------
fig.set_figwidth(10)
fig.set_figheight(5)
fig.suptitle(r"Max Preamp Decoupling and Max Gain vs NF, Varying $Z_\mathrm{amp}$")

# --- axes formatting -------------------------------------------------------
ax1.set_title("Max Decoupling vs NF")
ax1.set_xlabel("NF (dB)")
ax1.set_ylabel("Max Decoupling (dB)")
ax1.invert_yaxis()

ax2.set_title("Max Relative Gain vs NF")
ax2.set_xlabel("NF (dB)")
ax2.set_ylabel("Max Relative Gain (dB)")


matplotlib.rcParams.update({'font.size': 12}) # Set fonts. Source - https://stackoverflow.com/a/3900167


plt.tight_layout()
plt.show()

Below we set the amplifier input impedance **constant**, which is the amplifier input impedance when it is small-signal complex-conjugately matched. We use this amplifier impedance to calculate $\Gamma_\mathrm{out}$, pramplifier decoupling, and maximum gain. This is the unilateral-approximation version of computing the preamplifier decoupling vs noise figure tradeoff, as written at the end of §2.2 in reference \[1\]:
> When a preamplifier input impedance is insensitive to output load because of high reverse isolation or fixed load termination like 50 Ω, the constant-noise contours $Γ_\mathrm{out}(F)$ become circles, and Equation (16) has a relatively simple analytical form, as shown in Supporting Information A.

In [ ]:
# --- compute ---------------------------------------------------------------
min_I, _, _, max_G = amp.min_and_max_I_and_G_at_given_NF_for_constant_z_in_amp(dNF)

# --- plot ------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, num=1, clear=True)

(line1,) = ax1.plot(dNF + nf_min_dB, -20 * np.log10(min_I), linewidth=1, label=r"preamp decoupling (constant z_{in})")

ax1.plot(nf_min_dB, -20 * np.log10(min_I[0]),
        marker=".", markersize=15, color=line1.get_color(), label="_nolegend_")

ax1.annotate(
    f" {nf_min_dB:.2f} dB NF\n {-20 * np.log10(min_I[0]):.1f} dB Dcp",
    xy=(nf_min_dB, -20 * np.log10(min_I[0])),
    fontfamily="monospace", fontsize=12, va="center", ha="left",
)

(line2,) = ax2.plot(dNF + nf_min_dB, 10 * np.log10(max_G), linewidth=1, label=r"max relative gain (constant z_{in})")

ax2.plot(nf_min_dB, 10 * np.log10(max_G[0]),
        marker=".", markersize=15, color=line1.get_color(), label="_nolegend_")

ax2.annotate(
    f" {nf_min_dB:.2f} dB NF\n {10 * np.log10(max_G[0]):.1f} dB Gain",
    xy=(nf_min_dB, 10 * np.log10(max_G[0])),
    fontfamily="monospace", fontsize=12, va="center", ha="left",
)

# --- figure formatting -----------------------------------------------------
fig.set_figwidth(10)
fig.set_figheight(5)
fig.suptitle(r"Max Preamp Decoupling and Max Gain vs NF, Constant $Z_\mathrm{amp}$")

# --- axes formatting -------------------------------------------------------
ax1.set_xlabel("NF (dB)")
ax1.set_ylabel("Max Decoupling (dB)")
ax1.invert_yaxis()

ax2.set_xlabel("NF (dB)")
ax2.set_ylabel("Max Relative Gain (dB)")


matplotlib.rcParams.update({'font.size': 12}) # Source - https://stackoverflow.com/a/3900167


plt.tight_layout()
plt.show()

As you have seen the results are a little off from the varying-$Z_\mathrm{amp}$ version. This means the results are a little dependent on the choice of $Z_\mathrm{amp}$. I suggest you always use the varying-$Z_\mathrm{amp}$ version because:
- The varying-$Z_\mathrm{amp}$ version gives accurate results.
- You need a program to compute the preamplifier decoupling or relative gain vs noise figure trade-off anyway. 